In [1]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
import faiss
import pickle
import os
from pathlib import Path

print("=" * 60)
print("CBF MODEL TRAINING")
print("=" * 60)

# Load the movies DataFrame with combined_content
movies_df = pd.read_pickle("../models/cbf/movies_df.pkl")
print(f"\nLoaded {len(movies_df)} movies")
print(f"Columns: {movies_df.columns.tolist()}")
print(f"Sample combined_content (first 200 chars):")
print(movies_df["combined_content"].iloc[0][:200])

# Check for any null or empty content
empty_content = movies_df["combined_content"].str.len() == 0
print(f"\nMovies with empty content: {empty_content.sum()}")
if empty_content.sum() > 0:
    print("These are the movies without content (likely ID 401840):")
    print(movies_df[empty_content][['id', 'title']].head())

# Filter out empty content for training
movies_for_training = movies_df[~empty_content].copy()
print(f"\nTraining on {len(movies_for_training)} movies")

CBF MODEL TRAINING

Loaded 45424 movies
Columns: ['adult', 'belongs_to_collection', 'budget_x', 'genres_x', 'homepage', 'id', 'imdb_id', 'original_language', 'original_title', 'overview', 'popularity', 'poster_path', 'production_companies', 'production_countries', 'release_date', 'revenue', 'runtime', 'spoken_languages', 'status', 'tagline', 'title', 'video', 'vote_average', 'vote_count', 'release_year', 'genres_list', 'primary_genre', 'production_companies_list', 'cast_top5', 'directors', 'writers', 'plot_keywords', 'content_rating', 'director_name', 'actor_1_name', 'actor_2_name', 'actor_3_name', 'imdb_score', 'gross', 'movie_title', 'genres_y', 'budget_y', 'language', 'wiki_plot', 'bayesian_avg', 'rating_count', 'raw_avg', 'combined_content']
Sample combined_content (first 200 chars):
john lasseter john lasseter john lasseter john lasseter tom hanks tom hanks tom hanks tim allen tim allen rickles jim varney wallace shawn andrew stanton joel cohen joss whedon alec sokolow andrew sta


In [2]:

#  TF-IDF with exact parameters from spec

print("\n" + "=" * 60)
print("STEP 5.1: TF-IDF VECTORIZATION")
print("=" * 60)

tfidf_params = {
    'max_features': 75000,
    'ngram_range': (1, 2),
    'min_df': 3,
    'max_df': 0.85,
    'sublinear_tf': True,
    'token_pattern': r'(?u)\b[a-z_][a-z_]+\b',  # Allows underscores for compound names
    'analyzer': 'word'
}

print("Parameters:")
for k, v in tfidf_params.items():
    print(f"  {k:15}: {v}")

tfidf = TfidfVectorizer(**tfidf_params)

print("\nFitting TF-IDF on combined_content...")
X_tfidf = tfidf.fit_transform(movies_for_training['combined_content'])

print(f"\nTF-IDF matrix shape: {X_tfidf.shape}")
print(f"Vocabulary size: {len(tfidf.vocabulary_)}")
print(f"Sparsity: {X_tfidf.nnz / (X_tfidf.shape[0] * X_tfidf.shape[1]) * 100:.4f}%")

# Save the vectorizer
with open('../models/cbf/tfidf_vectorizer.pkl', 'wb') as f:
    pickle.dump(tfidf, f)
print("\n✓ Saved: ../models/cbf/tfidf_vectorizer.pkl")


STEP 5.1: TF-IDF VECTORIZATION
Parameters:
  max_features   : 75000
  ngram_range    : (1, 2)
  min_df         : 3
  max_df         : 0.85
  sublinear_tf   : True
  token_pattern  : (?u)\b[a-z_][a-z_]+\b
  analyzer       : word

Fitting TF-IDF on combined_content...

TF-IDF matrix shape: (45423, 75000)
Vocabulary size: 75000
Sparsity: 0.0794%

✓ Saved: ../models/cbf/tfidf_vectorizer.pkl


In [3]:

# Dimensionality reduction with TruncatedSVD
print("\n" + "=" * 60)
print("STEP 5.2: TRUNCATED SVD")
print("=" * 60)

# Try different K values as per spec
K_values = [50, 100, 150, 200, 300]
explained_variances = {}

for K in K_values:
    print(f"\nFitting SVD with K={K}...")
    svd = TruncatedSVD(n_components=K, n_iter=10, random_state=42)
    svd.fit(X_tfidf)
    explained_variances[K] = svd.explained_variance_ratio_.sum()
    print(f"  Explained variance: {explained_variances[K]:.4f}")

# Choose best K (smallest where explained variance > 0.5, or 200 as default)
best_K = None
for K in K_values:
    if explained_variances[K] > 0.5:
        best_K = K
        break

if best_K is None:
    best_K = 200
    print(f"\nNo K > 0.5, using default K={best_K}")

print(f"\nSelected K = {best_K}")

# Train final SVD with best_K
svd = TruncatedSVD(n_components=best_K, n_iter=10, random_state=42)
X_svd = svd.fit_transform(X_tfidf)

print(f"\nSVD matrix shape: {X_svd.shape}")
print(f"Final explained variance: {svd.explained_variance_ratio_.sum():.4f}")

# Save the SVD model
with open('../models/cbf/svd_model.pkl', 'wb') as f:
    pickle.dump(svd, f)
print("✓ Saved: ../models/cbf/svd_model.pkl")


STEP 5.2: TRUNCATED SVD

Fitting SVD with K=50...
  Explained variance: 0.0466

Fitting SVD with K=100...
  Explained variance: 0.0672

Fitting SVD with K=150...
  Explained variance: 0.0841

Fitting SVD with K=200...
  Explained variance: 0.0989

Fitting SVD with K=300...
  Explained variance: 0.1244

No K > 0.5, using default K=200

Selected K = 200

SVD matrix shape: (45423, 200)
Final explained variance: 0.0989
✓ Saved: ../models/cbf/svd_model.pkl


In [4]:
# Create ID mappings for FAISS

print("\n" + "=" * 60)
print("STEP: CREATING ID MAPPINGS")
print("=" * 60)

# Create mappings between TMDB ID and matrix index
movie_id_to_idx = {}
idx_to_movie_id = {}

for idx, (original_idx, row) in enumerate(movies_for_training.iterrows()):
    tmdb_id = row['id']
    movie_id_to_idx[tmdb_id] = idx
    idx_to_movie_id[idx] = tmdb_id

print(f"Created mappings for {len(movie_id_to_idx)} movies")

# Save mappings
with open('../models/cbf/movie_id_to_idx.pkl', 'wb') as f:
    pickle.dump(movie_id_to_idx, f)

with open('../models/cbf/idx_to_movie_id.pkl', 'wb') as f:
    pickle.dump(idx_to_movie_id, f)

print("✓ Saved: movie_id_to_idx.pkl and idx_to_movie_id.pkl")


STEP: CREATING ID MAPPINGS
Created mappings for 45423 movies
✓ Saved: movie_id_to_idx.pkl and idx_to_movie_id.pkl


In [5]:
# Build FAISS index for fast similarity search

print("\n" + "=" * 60)
print("STEP 5.3: FAISS INDEX CONSTRUCTION")
print("=" * 60)

# Convert to float32 for FAISS
X_svd_float32 = X_svd.astype('float32')

# Normalize to unit length (so inner product = cosine similarity)
faiss.normalize_L2(X_svd_float32)

# Create index
d = X_svd_float32.shape[1]  # dimension
index = faiss.IndexFlatIP(d)  # Inner Product = Cosine Similarity after normalization
index.add(X_svd_float32)

print(f"FAISS index created with {index.ntotal} vectors of dimension {d}")
print(f"Index type: {type(index).__name__}")

# Save the index
faiss.write_index(index, '../models/cbf/faiss_index.bin')
print("✓ Saved: ../models/cbf/faiss_index.bin")

# Quick test
test_idx = 0
query_vector = X_svd_float32[test_idx:test_idx+1]
D, I = index.search(query_vector, k=13)

print(f"\nTest query (first movie):")
print(f"  Query movie ID: {idx_to_movie_id[test_idx]}")
print(f"  Top result (should be itself): ID {idx_to_movie_id[I[0][0]]}, similarity {D[0][0]:.4f}")
print(f"  Next 5 recommendations: {[idx_to_movie_id[i] for i in I[0][1:6]]}")


STEP 5.3: FAISS INDEX CONSTRUCTION
FAISS index created with 45423 vectors of dimension 200
Index type: IndexFlatIP
✓ Saved: ../models/cbf/faiss_index.bin

Test query (first movie):
  Query movie ID: 862
  Top result (should be itself): ID 862, similarity 1.0000
  Next 5 recommendations: [863, 213121, 10193, 175574, 130925]


In [6]:
# Create genre vectors for demographic blending (Section 4.3)

print("\n" + "=" * 60)
print("STEP 4.3: GENRE VECTORS FOR DEMOGRAPHIC BLENDING")
print("=" * 60)

# Get all unique genres
all_genres = set()
for genres in movies_for_training['genres_list']:
    if isinstance(genres, list):
        all_genres.update(genres)

print(f"Found {len(all_genres)} unique genres: {sorted(all_genres)[:10]}...")

# Compute mean SVD vector for each genre
genre_vectors = {}

for genre in all_genres:
    # Find movies that have this genre
    genre_mask = movies_for_training['genres_list'].apply(
        lambda x: isinstance(x, list) and genre in x
    )
    
    if genre_mask.sum() > 0:
        # Get their SVD vectors
        genre_indices = movies_for_training[genre_mask].index
        genre_indices_in_svd = [movie_id_to_idx[movies_for_training.loc[idx]['id']] 
                                for idx in genre_indices]
        
        genre_vectors[genre] = X_svd[genre_indices_in_svd].mean(axis=0)
        print(f"  {genre:20}: {genre_mask.sum():5d} movies")

# Save genre vectors
with open('../models/cbf/genre_vectors.pkl', 'wb') as f:
    pickle.dump(genre_vectors, f)
print("\n✓ Saved: ../models/cbf/genre_vectors.pkl")


STEP 4.3: GENRE VECTORS FOR DEMOGRAPHIC BLENDING
Found 20 unique genres: ['Action', 'Adventure', 'Animation', 'Comedy', 'Crime', 'Documentary', 'Drama', 'Family', 'Fantasy', 'Foreign']...
  Family              :  2767 movies
  History             :  1398 movies
  Horror              :  4668 movies
  Mystery             :  2464 movies
  Documentary         :  3930 movies
  Animation           :  1931 movies
  Crime               :  4300 movies
  Science Fiction     :  3043 movies
  Action              :  6592 movies
  TV Movie            :   766 movies
  War                 :  1322 movies
  Romance             :  6728 movies
  Western             :  1042 movies
  Fantasy             :  2309 movies
  Foreign             :  1619 movies
  Thriller            :  7617 movies
  Comedy              : 13173 movies
  Adventure           :  3490 movies
  Music               :  1596 movies
  Drama               : 20242 movies

✓ Saved: ../models/cbf/genre_vectors.pkl


In [7]:

# Verify all artifacts are saved

print("\n" + "=" * 60)
print("CBF MODEL ARTIFACTS - FINAL CHECK")
print("=" * 60)

expected_files = [
    ('../models/cbf/tfidf_vectorizer.pkl', 'TF-IDF Vectorizer'),
    ('../models/cbf/svd_model.pkl', 'SVD Model'),
    ('../models/cbf/faiss_index.bin', 'FAISS Index'),
    ('../models/cbf/movie_id_to_idx.pkl', 'Movie ID → Index'),
    ('../models/cbf/idx_to_movie_id.pkl', 'Index → Movie ID'),
    ('../models/cbf/genre_vectors.pkl', 'Genre Vectors'),
]

all_good = True
for filepath, description in expected_files:
    if os.path.exists(filepath):
        size_mb = os.path.getsize(filepath) / (1024 * 1024)
        print(f"✓ {description:25}: {filepath} ({size_mb:.2f} MB)")
    else:
        print(f"✗ {description:25}: MISSING - {filepath}")
        all_good = False

print("\n" + "=" * 60)
if all_good:
    print("✅ ALL CBF MODEL ARTIFACTS CREATED SUCCESSFULLY")
    print("✅ Ready for Section 6: MLflow Tracking")
else:
    print("❌ ERROR: Some artifacts missing - check above")


CBF MODEL ARTIFACTS - FINAL CHECK
✓ TF-IDF Vectorizer        : ../models/cbf/tfidf_vectorizer.pkl (2.57 MB)
✓ SVD Model                : ../models/cbf/svd_model.pkl (114.45 MB)
✓ FAISS Index              : ../models/cbf/faiss_index.bin (34.66 MB)
✓ Movie ID → Index         : ../models/cbf/movie_id_to_idx.pkl (0.30 MB)
✓ Index → Movie ID         : ../models/cbf/idx_to_movie_id.pkl (0.30 MB)
✓ Genre Vectors            : ../models/cbf/genre_vectors.pkl (0.03 MB)

✅ ALL CBF MODEL ARTIFACTS CREATED SUCCESSFULLY
✅ Ready for Section 6: MLflow Tracking


In [9]:
%pip install mlflow

  Using cached docker-7.1.0-py3-none-any.whl.metadata (3.8 kB)
  Using cached graphene-3.4.3-py2.py3-none-any.whl.metadata (6.9 kB)
  Using cached waitress-3.0.2-py3-none-any.whl.metadata (5.8 kB)
  Using cached gitpython-3.1.46-py3-none-any.whl.metadata (13 kB)
  Using cached sqlparse-0.5.5-py3-none-any.whl.metadata (4.7 kB)
  Using cached cffi-2.0.0-cp311-cp311-win_amd64.whl.metadata (2.6 kB)
  Using cached gitdb-4.0.12-py3-none-any.whl.metadata (1.2 kB)
  Using cached smmap-5.0.2-py3-none-any.whl.metadata (4.3 kB)
  Using cached pyasn1_modules-0.4.2-py3-none-any.whl.metadata (3.5 kB)
  Using cached rsa-4.9.1-py3-none-any.whl.metadata (5.6 kB)
  Using cached graphql_relay-3.2.0-py3-none-any.whl.metadata (12 kB)
  Using cached pycparser-3.0-py3-none-any.whl.metadata (8.2 kB)
   ---------------------------------------- 0.0/10.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/10.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/10.2 MB ? eta -:--:--
   

In [ ]:
import mlflow
import mlflow.sklearn
from datetime import datetime
import os
from mlflow.models.signature import infer_signature
import hashlib


# Set MLflow experiment
mlflow.set_experiment("cbf_content_based_filtering")

with mlflow.start_run(run_name=f"cbf_tfidf_svd_faiss_{datetime.now().strftime('%Y%m%d_%H%M')}"):

      # --- DATA SIGNATURE ---
    sample_input = movie_content["combined_features"].iloc[:5]
    sample_output = tfidf.transform(sample_input)
    signature = infer_signature(sample_input, sample_output)

    # --- DATA HASH ---
    mlflow.sklearn.log_model(
        tfidf,
        "tfidf_vectorizer",
        signature=signature
    )
    data_hash = hashlib.md5(
        open("../data/processed/movie_content.csv", "rb").read()
    ).hexdigest()
    mlflow.log_param("movie_content_md5", data_hash)


    # --- PARAMETERS ---
    mlflow.log_param("tfidf_max_features", 75000)
    mlflow.log_param("tfidf_ngram_range", "(1,2)")
    mlflow.log_param("tfidf_min_df", 3)
    mlflow.log_param("tfidf_max_df", 0.85)
    mlflow.log_param("tfidf_sublinear_tf", True)
    mlflow.log_param("svd_n_components", 200)
    mlflow.log_param("svd_n_iter", 10)
    mlflow.log_param("n_movies", 45423)
    mlflow.log_param("faiss_index_type", "IndexFlatIP")
    mlflow.log_param("demographic_blending_beta", 0.15)
    
    # --- METRICS ---
    # mlflow.log_metric("svd_explained_variance_ratio", 0.0989)
    # mlflow.log_metric("tfidf_vocab_size", 75000)
    # mlflow.log_metric("faiss_index_size_mb", 34.66)
    # mlflow.log_metric("svd_model_size_mb", 114.45)
    mlflow.log_metric(
    "svd_explained_variance_ratio",
    float(svd.explained_variance_ratio_.sum())
    )

    mlflow.log_metric(
        "tfidf_vocab_size",
        len(tfidf.vocabulary_)
    )

    mlflow.log_metric(
        "faiss_index_size_mb",
        os.path.getsize("../models/cbf/faiss_index.bin") / (1024 * 1024)
    )

    mlflow.log_metric(
        "svd_model_size_mb",
        os.path.getsize("../models/cbf/svd_model.pkl") / (1024 * 1024)
    )
    
    # --- ARTIFACTS ---
    # mlflow.log_artifact("../models/cbf/tfidf_vectorizer.pkl")
    # mlflow.log_artifact("../models/cbf/svd_model.pkl")
    mlflow.log_artifact("../models/cbf/faiss_index.bin")
    mlflow.log_artifact("../models/cbf/movie_id_to_idx.pkl")
    mlflow.log_artifact("../models/cbf/idx_to_movie_id.pkl")
    mlflow.log_artifact("../models/cbf/genre_vectors.pkl")
    mlflow.log_artifact("../data/processed/movie_content.csv")
    
    # --- MODEL LOGGING ---
    mlflow.sklearn.log_model(tfidf, "tfidf_vectorizer")
    mlflow.sklearn.log_model(svd, "svd_model")
    
    # --- TAGS ---
    mlflow.set_tag("model_stage", "cbf_only")
    mlflow.set_tag("training_date", datetime.now().isoformat())
    mlflow.set_tag("status", "production_ready")

print("✓ MLflow run completed")

c:\Users\ADEGOKE\anaconda3\envs\Recommender\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026/02/24 01:42:42 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/02/24 01:42:42 INFO mlflow.store.db.utils: Updating database tables
2026/02/24 01:42:52 INFO mlflow.tracking.fluent: Experiment with name 'cbf_content_based_filtering' does not exist. Creating a new experiment.
2026/02/24 01:42:57 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/02/24 01:42:57 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. 

✓ MLflow run completed


In [1]:
!mlflow ui --backend-store-uri ./mlruns

^C


In [ ]:
# import numpy as np
# print(np.__version__)  # should be 2.0.2

1.26.4
